In [25]:
import pandas as pd

df = pd.read_excel('Areas-of-interest-POIs\sample_1000-updated.xlsx')
df.head()

,mid_label,mid_label_new,bosserhof_class_clean,bosserhof_class_clean.1,osm_names,gml_id,target_taz,volume_m3,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,"['Leisure', 'Workers']",NaN,restaurants gastronomy,NaN,['Enoteca Vetrone'],236928,Rühen 7_472,1554.252341,NaN,NaN,NaN,NaN
1,['Workers'],NaN,normal office,NaN,NaN,535772,WOB Ehmen 2_289,453.347111,NaN,NaN,NaN,NaN
2,['Workers'],NaN,industrial operations production,NaN,NaN,477334,Gifhorn 04_416,727.942206,NaN,NaN,NaN,NaN
3,"['Retail_Daily', 'Workers', 'Retail_Non-Daily']","['Workers', 'Retail_Non-Daily']",retail small scale,NaN,"[""Deutsche Bank;Ernsting's family""]",388629,Goslar 18_565,17437.047953,NaN,NaN,NaN,NaN
4,['Workers'],NaN,normal office,NaN,NaN,81399,Bad Harzburg Schlewecke_518,248.752010,NaN,NaN,NaN,NaN


In [26]:
import re
import ast
from collections import Counter

import pandas as pd
from openpyxl import load_workbook


INPUT_FILE = "Areas-of-interest-POIs\\sample_1000-updated.xlsx"
N_ROWS = 349

GREEN = "FFC6EFCE"
RED = "FFFFC7CE"
YELLOW = "FFFFEB9C"


def clean_label(value):
    if value is None or str(value).strip() == "":
        return ""

    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def parse_labels(value):
    if value is None or str(value).strip() == "":
        return []

    value = str(value).strip()

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [clean_label(x) for x in parsed if clean_label(x)]
        return [clean_label(parsed)]
    except Exception:
        return [clean_label(x) for x in value.split(",") if clean_label(x)]


def get_colour(cell):
    colour = cell.fill.fgColor.rgb
    return colour if colour else "NO_COLOR"


def colour_status(colour):
    if colour == GREEN:
        return "correct"
    elif colour == RED:
        return "incorrect"
    elif colour == YELLOW:
        return "uncertain"
    else:
        return "unknown"


# =========================
# LOAD EXCEL
# =========================

wb = load_workbook(INPUT_FILE)
ws = wb.active

headers = {
    ws.cell(row=1, column=col).value: col
    for col in range(1, ws.max_column + 1)
}

mid_label_col = headers["mid_label"]
mid_label_new_col = headers["mid_label_new"]

bosserhof_col = headers["bosserhof_class_clean"]
bosserhof_clean_col = headers["bosserhof_class_clean"]

gml_id_col = headers.get("gml_id")


# =========================
# PROCESS FIRST 150 ROWS
# =========================

processed_rows = []

summary_counter = Counter()
missed_label_counter = Counter()
extra_label_counter = Counter()

for row in range(2, N_ROWS + 2):

    gml_id = ws.cell(row, gml_id_col).value if gml_id_col else None

    predicted_mid_labels = parse_labels(ws.cell(row, mid_label_col).value)
    validator_mid_labels = parse_labels(ws.cell(row, mid_label_new_col).value)

    mid_status = colour_status(get_colour(ws.cell(row, mid_label_new_col)))

    predicted_set = set(predicted_mid_labels)
    validator_set = set(validator_mid_labels)

    if mid_status == "incorrect":
        missed_labels = sorted(validator_set - predicted_set)
        extra_labels = sorted(predicted_set - validator_set)
    else:
        missed_labels = []
        extra_labels = []

    for label in missed_labels:
        missed_label_counter[label] += 1

    for label in extra_labels:
        extra_label_counter[label] += 1

    summary_counter[f"mid_label_{mid_status}"] += 1

    predicted_bosserhof = clean_label(ws.cell(row, bosserhof_col).value)
    validator_bosserhof = clean_label(ws.cell(row, bosserhof_clean_col).value)

    bosserhof_status = colour_status(get_colour(ws.cell(row, bosserhof_clean_col)))

    summary_counter[f"bosserhof_{bosserhof_status}"] += 1

    final_bosserhof = (
        validator_bosserhof
        if bosserhof_status == "incorrect" and validator_bosserhof
        else predicted_bosserhof
    )

    processed_rows.append({
        "gml_id": gml_id,

        "mid_label_original": ws.cell(row, mid_label_col).value,
        "mid_label_clean": predicted_mid_labels,

        "mid_label_new_original": ws.cell(row, mid_label_new_col).value,
        "mid_label_new_clean": validator_mid_labels,

        "mid_label_status": mid_status,
        "missed_labels_by_llm": missed_labels,
        "extra_labels_by_llm": extra_labels,
        "n_missed_labels": len(missed_labels),
        "n_extra_labels": len(extra_labels),

        "bosserhof_class_original": ws.cell(row, bosserhof_col).value,
        "bosserhof_class_cleaned_predicted": predicted_bosserhof,

        "bosserhof_class_clean_original": ws.cell(row, bosserhof_clean_col).value,
        "bosserhof_class_cleaned_validator": validator_bosserhof,

        "bosserhof_status": bosserhof_status,
        "final_bosserhof_class": final_bosserhof,
    })


# Main dataset for further work
processed_df = pd.DataFrame(processed_rows)

# Optional summary datasets
summary_df = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in summary_counter.items()
]).sort_values("metric")

missed_labels_df = pd.DataFrame([
    {"label": key, "count": value}
    for key, value in missed_label_counter.items()
]).sort_values("count", ascending=False)

extra_labels_df = pd.DataFrame([
    {"label": key, "count": value}
    for key, value in extra_label_counter.items()
]).sort_values("count", ascending=False)


processed_df

,gml_id,mid_label_original,mid_label_clean,mid_label_new_original,mid_label_new_clean,mid_label_status,missed_labels_by_llm,extra_labels_by_llm,n_missed_labels,n_extra_labels,bosserhof_class_original,bosserhof_class_cleaned_predicted,bosserhof_class_clean_original,bosserhof_class_cleaned_validator,bosserhof_status,final_bosserhof_class
0,236928,"['Leisure', 'Workers']","[leisure, workers]",None,[],correct,[],[],0,0,None,,None,,correct,
1,535772,['Workers'],[workers],None,[],uncertain,[],[],0,0,None,,None,,uncertain,
2,477334,['Workers'],[workers],None,[],uncertain,[],[],0,0,None,,None,,uncertain,
3,388629,"['Retail_Daily', 'Workers', 'Retail_Non-Daily']","[retail daily, workers, retail non daily]","['Workers', 'Retail_Non-Daily']","[workers, retail non daily]",incorrect,[],[retail daily],0,1,None,,None,,correct,
4,81399,['Workers'],[workers],None,[],uncertain,[],[],0,0,None,,None,,uncertain,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,205868,['Workers'],[workers],None,[],uncertain,[],[],0,0,None,,None,,uncertain,
345,212017,['Workers'],[workers],None,[],correct,[],[],0,0,None,,None,,correct,
346,469652,['Workers'],[workers],None,[],uncertain,[],[],0,0,None,,None,,uncertain,
347,603101,['Workers'],[workers],['Retail-non-daily'],[retail non daily],incorrect,[retail non daily],[workers],1,1,None,,None,,correct,


In [27]:
processed_df = processed_df.drop(columns=['mid_label_original', 'mid_label_new_original',
                                           'bosserhof_class_cleaned_predicted', 'bosserhof_class_clean_original',
                                           'bosserhof_class_cleaned_validator', 'final_bosserhof_class'],axis=1)

processed_df.head()

,gml_id,mid_label_clean,mid_label_new_clean,mid_label_status,missed_labels_by_llm,extra_labels_by_llm,n_missed_labels,n_extra_labels,bosserhof_class_original,bosserhof_status
0,236928,"[leisure, workers]",[],correct,[],[],0,0,None,correct
1,535772,[workers],[],uncertain,[],[],0,0,None,uncertain
2,477334,[workers],[],uncertain,[],[],0,0,None,uncertain
3,388629,"[retail daily, workers, retail non daily]","[workers, retail non daily]",incorrect,[],[retail daily],0,1,None,correct
4,81399,[workers],[],uncertain,[],[],0,0,None,uncertain


In [28]:
processed_df[processed_df['mid_label_status'] == 'uncertain']['mid_label_clean'].value_counts()

mid_label_clean
[workers]                                    204
[workers, retail daily]                        6
[workers, retail non daily, retail daily]      3
[leisure]                                      3
[retail non daily]                             2
[retail daily]                                 2
[workers, leisure]                             2
[retail non daily, workers]                    2
[retail daily, workers]                        1
[workers, retail non daily]                    1
[retail non daily, retail daily, workers]      1
Name: count, dtype: int64

In [29]:
processed_df['bosserhof_status'].value_counts()

bosserhof_status
uncertain    226
correct      100
incorrect     23
Name: count, dtype: int64

In [30]:
processed_df['mid_label_status'].value_counts() 

mid_label_status
uncertain    227
correct       74
incorrect     48
Name: count, dtype: int64

In [31]:
correct_rows = processed_df[
    processed_df["mid_label_status"].astype(str).str.lower().eq("correct")
].copy()

incorrect_rows = processed_df[
    processed_df["mid_label_status"].astype(str).str.lower().eq("incorrect")
].copy()

print("Correct rows:", len(correct_rows))
print("Incorrect rows:", len(incorrect_rows))

Correct rows: 74
Incorrect rows: 48


In [32]:
incorrect_rows.head()

,gml_id,mid_label_clean,mid_label_new_clean,mid_label_status,missed_labels_by_llm,extra_labels_by_llm,n_missed_labels,n_extra_labels,bosserhof_class_original,bosserhof_status
3,388629,"[retail daily, workers, retail non daily]","[workers, retail non daily]",incorrect,[],[retail daily],0,1,None,correct
6,36228,"[retail daily, retail non daily, university, l...","[retail daily, retail non daily, university, w...",incorrect,[],[leisure],0,1,None,correct
8,389444,"[workers, retail daily]","[workers, retail daily, retail non daily]",incorrect,[retail non daily],[],1,0,None,correct
9,453160,"[retail non daily, retail daily]",[retail non daily],incorrect,[],[retail daily],0,1,business oriented services,incorrect
10,21469,[workers],"[workers, leisure]",incorrect,[leisure],[],1,0,entertainment culture,incorrect


In [33]:
print(incorrect_rows['n_missed_labels'].sum())
print(incorrect_rows['n_extra_labels'].sum())

35
39


In [34]:
from collections import Counter

all_labels = [
    label
    for labels in incorrect_rows["mid_label_clean"].apply(parse_labels)
    for label in labels
]

label_counts = Counter(all_labels)

unique_label_counts = pd.DataFrame(
    label_counts.items(),
    columns=["label", "count"]
).sort_values("count", ascending=False)

print("Total unique labels:", len(label_counts))

unique_label_counts

Total unique labels: 6


,label,count
1,workers,36
0,retail daily,30
2,retail non daily,13
4,leisure,3
3,university,1
5,kindergarten,1


In [35]:
# Unique label names and their frequency in correct_rows["mid_label_clean"]
from collections import Counter

all_labels_correct = [
    label
    for labels in correct_rows["mid_label_clean"].apply(parse_labels)
    for label in labels
]

label_counts_correct = Counter(all_labels_correct)

unique_label_counts_correct = pd.DataFrame(
    label_counts_correct.items(),
    columns=["label", "count"]
).sort_values("count", ascending=False)

print("Total unique labels:", len(label_counts_correct))

unique_label_counts_correct

Total unique labels: 7


,label,count
1,workers,72
0,leisure,25
2,retail daily,15
3,retail non daily,11
4,school,2
6,university,2
5,kindergarten,1


In [36]:
print(correct_rows['n_missed_labels'].sum())
print(correct_rows['n_extra_labels'].sum())

0
0
